## MLlib & Spark ML Pipelines

Spark's machine learning library — **MLlib** — brings ML to large-scale, distributed data. Instead of collecting data to a single machine for training, Spark trains models in parallel across the cluster using the same DataFrames and transformations you already know.

In a banking context, MLlib powers:
- **Fraud detection** — classify card transactions as fraudulent or legitimate
- **Credit risk scoring** — predict default probability for loan applicants
- **Customer segmentation** — cluster customers by spend behaviour
- **Churn prediction** — identify customers likely to close accounts

In this notebook you will learn:
- The difference between the old RDD-based API and the current DataFrame-based `spark.ml` API
- The three core abstractions: **Transformer**, **Estimator**, and **Pipeline**
- Feature engineering: `StringIndexer`, `OneHotEncoder`, `VectorAssembler`, `StandardScaler`
- Building an end-to-end **fraud detection classification pipeline**
- Evaluating models with `BinaryClassificationEvaluator` and `MulticlassClassificationEvaluator`
- Hyperparameter tuning with `CrossValidator` and `ParamGridBuilder`
- Building a **loan interest rate regression pipeline**
- Saving and loading ML pipelines

### The Two APIs — `spark.mllib` vs `spark.ml`

| | `spark.mllib` (legacy) | `spark.ml` (current) |
|---|---|---|
| Data abstraction | RDD | DataFrame |
| Status | Maintenance only since Spark 2.0 | Active — use this |
| Pipelines | No | Yes |
| Integration | Manual wiring | Composable pipeline stages |

Always use `spark.ml` for new code. The import path is `pyspark.ml.*`.

### Core Abstractions

```
Transformer                        Estimator
───────────                        ─────────
Has transform(df) → df             Has fit(df) → Model (a Transformer)
No training needed                 Learns parameters from data

Examples:                          Examples:
  OneHotEncoder                      StringIndexer
  VectorAssembler                    StandardScaler
  Binarizer                          LogisticRegression
  SQLTransformer                     RandomForestClassifier
```

**Pipeline** chains Transformers and Estimators into a single object:
- `pipeline.fit(train_df)` → fits every Estimator stage in order → returns a `PipelineModel`
- `pipeline_model.transform(df)` → runs every stage's `transform()` in order

```
Pipeline.fit(train):
  StringIndexer.fit(train) → StringIndexerModel.transform(train)
      ↓
  OneHotEncoder.fit(...)  → OneHotEncoderModel.transform(...)
      ↓
  VectorAssembler.transform(...)           ← already a Transformer
      ↓
  LogisticRegression.fit(...)  → LogisticRegressionModel.transform(...)
      ↓
  PipelineModel  (a reusable Transformer)
```

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType,
    DecimalType, DoubleType, IntegerType, DateType, TimestampType
)
from pyspark.sql.functions import col, hour, count

spark = (
    SparkSession.builder
    .appName("MLlibSparkMLPipelines")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

DATA = os.path.join(os.path.dirname(os.path.abspath("21-mllib-spark-ml-pipelines.ipynb")), "data")

card_txns = spark.read.schema(StructType([
    StructField("txn_id",      StringType(),      False),
    StructField("card_id",     StringType(),      False),
    StructField("customer_id", StringType(),      False),
    StructField("amount",      DecimalType(18,2), False),
    StructField("merchant",    StringType(),      True),
    StructField("category",    StringType(),      True),
    StructField("txn_ts",      TimestampType(),   False),
    StructField("status",      StringType(),      False),
    StructField("is_fraud",    BooleanType(),     False),
])).parquet(f"{DATA}/card_transactions")

loan_accounts = spark.read.schema(StructType([
    StructField("loan_id",       StringType(),      False),
    StructField("customer_id",   StringType(),      False),
    StructField("loan_type",     StringType(),      False),
    StructField("principal",     DecimalType(18,2), False),
    StructField("interest_rate", DoubleType(),      False),
    StructField("tenure_months", IntegerType(),     False),
    StructField("disbursed_on",  DateType(),        False),
    StructField("status",        StringType(),      False),
])).json(f"{DATA}/loan_accounts")

print(f"card_txns    : {card_txns.count():>6,} rows")
print(f"loan_accounts: {loan_accounts.count():>6,} rows")

## Feature Engineering

ML algorithms require **numeric input**. Feature engineering transforms raw DataFrame columns into the numeric vectors that algorithms expect. The four most important transformers are:

| Transformer | Input | Output | Purpose |
|---|---|---|---|
| `StringIndexer` | String column | Double index | Encodes category labels as integers |
| `OneHotEncoder` | Index column | Sparse vector | Converts index to binary vector (no ordinal relationship) |
| `VectorAssembler` | Multiple columns | Single vector | Combines all features into one `features` column |
| `StandardScaler` | Vector column | Scaled vector | Normalises to zero mean / unit std |

### StringIndexer — Encoding Categorical Columns

`StringIndexer` maps each distinct string value to a numeric index. The most frequent value gets index `0`, the second most frequent gets `1`, and so on. `handleInvalid="keep"` assigns an extra index to any value seen at transform time that was not in the training data.

In [ ]:
from pyspark.ml.feature import StringIndexer

category_indexer = StringIndexer(
    inputCol="category",
    outputCol="categoryIdx",
    handleInvalid="keep",   # unseen values → extra index (not an error)
)

# fit() scans the data and learns the label → index mapping
indexer_model = category_indexer.fit(card_txns)

print("Learned label → index mapping:")
for idx, label in enumerate(indexer_model.labels):
    print(f"  {idx}  {label}")

# transform() applies the mapping to produce a new column
indexed = indexer_model.transform(card_txns)
indexed.select("category", "categoryIdx").distinct().orderBy("categoryIdx").show()

### OneHotEncoder — Removing Ordinal Relationships

Raw numeric indices carry an implicit ordering (index 2 appears "greater than" index 1). For most ML algorithms, this is wrong — `FOOD` is not greater than `TRAVEL`. `OneHotEncoder` converts each index to a **sparse binary vector** where only one position is `1.0` and all others are `0.0`.

For N distinct categories, the vector has length **N−1** (the last category is dropped as the reference level to avoid perfect multicollinearity in linear models).

In [ ]:
from pyspark.ml.feature import OneHotEncoder

encoder = OneHotEncoder(
    inputCol="categoryIdx",
    outputCol="categoryVec",
    dropLast=True,   # default — drops the last category to avoid multicollinearity
)

# OneHotEncoder is an Estimator in Spark 3 (fit() learns the category count)
encoder_model = encoder.fit(indexed)
encoded = encoder_model.transform(indexed)

print(f"Number of categories : {len(indexer_model.labels)}")
print(f"OHE vector length    : {len(indexer_model.labels) - 1}  (N-1, dropLast=True)")
print()
encoded.select("category", "categoryIdx", "categoryVec").distinct().orderBy("categoryIdx").show(truncate=False)

### VectorAssembler — One Feature Vector per Row

ML algorithms in `spark.ml` expect all features in a **single vector column** named `features` by convention. `VectorAssembler` takes multiple columns — numeric scalars, boolean values, OHE sparse vectors — and concatenates them into one dense or sparse vector per row.

In [ ]:
from pyspark.ml.feature import VectorAssembler

# Add numeric columns needed for assembly
demo_df = (
    encoded
    .withColumn("amount_f",  col("amount").cast("double"))
    .withColumn("txn_hour",  hour("txn_ts").cast("double"))
)

assembler = VectorAssembler(
    inputCols=["amount_f", "txn_hour", "categoryVec"],
    outputCol="features",
    handleInvalid="skip",   # skip rows with null in any input column
)

# VectorAssembler is a pure Transformer — no fit() needed
assembled = assembler.transform(demo_df)
assembled.select("amount_f", "txn_hour", "categoryVec", "features").show(5, truncate=False)

print("feature vector: [amount_f, txn_hour, categoryVec...]")
print(f"vector size: {len(assembled.select('features').first()[0])}")

### StandardScaler — Normalising Feature Magnitudes

Some algorithms — logistic regression, SVM, k-means — are sensitive to the absolute magnitude of features. A `amount` column ranging from 0 to 50,000 will dominate an `txn_hour` column ranging from 0 to 23. `StandardScaler` normalises each feature to **zero mean** (`withMean=True`) and **unit standard deviation** (`withStd=True`).

**Note**: `withMean=True` requires a **dense** input vector. OHE vectors are sparse — do not use `withMean=True` on them. Scale only the numeric (dense) portion, or use `withStd=True, withMean=False` on the combined feature vector.

In [ ]:
from pyspark.ml.feature import StandardScaler

# Scale only the two numeric features (dense) — not the OHE sparse vector
num_assembler = VectorAssembler(
    inputCols=["amount_f", "txn_hour"],
    outputCol="numeric_features"
)
num_df = num_assembler.transform(demo_df)

scaler = StandardScaler(
    inputCol="numeric_features",
    outputCol="scaled_numeric",
    withStd=True,
    withMean=True,   # OK — numeric_features is dense
)
scaler_model = scaler.fit(num_df)   # fit() computes mean and std per feature
scaled = scaler_model.transform(num_df)

scaled.select("numeric_features", "scaled_numeric").show(5, truncate=False)

print(f"Mean   (amount, hour): {scaler_model.mean.toArray().round(4)}")
print(f"StdDev (amount, hour): {scaler_model.std.toArray().round(4)}")
print("After scaling: mean ≈ 0, std ≈ 1 for each feature")

## Classification — Fraud Detection Pipeline

The `is_fraud` column in `card_transactions` is the label for a binary classification task. The goal is to predict whether a transaction is fraudulent based on the transaction amount, the hour of day, and the spending category.

**Feature strategy:**
- `amount` → cast to `double`
- `txn_ts` → extract `hour` as a `double`
- `category` → `StringIndexer` → `OneHotEncoder` → sparse vector
- Combine with `VectorAssembler`

**Label**: `is_fraud` cast to `double` (0.0 = legitimate, 1.0 = fraud)

**Algorithm**: `LogisticRegression` — the standard baseline for binary classification. It has `standardization=True` by default, which normalises features internally, so a separate `StandardScaler` stage is optional.

In [ ]:
# Prepare the fraud classification dataset
fraud_df = (
    card_txns
    .withColumn("label",    col("is_fraud").cast("double"))
    .withColumn("amount_f", col("amount").cast("double"))
    .withColumn("txn_hour", hour("txn_ts").cast("double"))
    .select("label", "amount_f", "txn_hour", "category")
    .dropna()
)

print("Class distribution:")
fraud_df.groupBy("label").count().orderBy("label").show()

total      = fraud_df.count()
fraud_count = fraud_df.filter("label = 1").count()
print(f"Total rows : {total:,}")
print(f"Fraud rows : {fraud_count:,}  ({fraud_count / total * 100:.1f}%)")
print(f"Legit rows : {total - fraud_count:,}  ({(total - fraud_count) / total * 100:.1f}%)")
print("\n(Fraud detection is always highly imbalanced — AUC-ROC is more informative than raw accuracy)")

### Building the End-to-End Pipeline

Chaining all feature engineering and the model into a single `Pipeline` keeps preprocessing and model training reproducible and serialisable. When you save a `PipelineModel`, all stage parameters are saved together.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression

# Stage 1: encode the category string to a numeric index
cat_idx = StringIndexer(inputCol="category", outputCol="categoryIdx", handleInvalid="keep")

# Stage 2: convert the index to a one-hot vector
cat_ohe = OneHotEncoder(inputCol="categoryIdx", outputCol="categoryVec")

# Stage 3: assemble all features into one vector
feat_asm = VectorAssembler(
    inputCols=["amount_f", "txn_hour", "categoryVec"],
    outputCol="features",
    handleInvalid="skip",
)

# Stage 4: logistic regression classifier
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    regParam=0.01,          # L2 regularisation strength
    standardization=True,   # normalises features internally (default True)
)

pipeline = Pipeline(stages=[cat_idx, cat_ohe, feat_asm, lr])

# Train / test split — reproducible with seed
train, test = fraud_df.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train.count():,}   Test: {test.count():,}")

# fit() runs Estimator.fit() for each stage in sequence
model = pipeline.fit(train)
print("\nFitted stages:", [type(s).__name__ for s in model.stages])

# transform() applies each stage's transform() to the test data
predictions = model.transform(test)
predictions.select("label", "probability", "prediction").show(10, truncate=False)

### Model Evaluation

| Evaluator | Metric | Use when |
|---|---|---|
| `BinaryClassificationEvaluator` | `areaUnderROC` (AUC) | Binary classification, imbalanced classes |
| `BinaryClassificationEvaluator` | `areaUnderPR` | Precision-recall tradeoff matters |
| `MulticlassClassificationEvaluator` | `accuracy` | Balanced classes |
| `MulticlassClassificationEvaluator` | `f1` | Harmonic mean of precision and recall |
| `RegressionEvaluator` | `rmse`, `mae`, `r2` | Regression tasks |

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

auc_eval = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
)
acc_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
f1_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)

auc = auc_eval.evaluate(predictions)
acc = acc_eval.evaluate(predictions)
f1  = f1_eval.evaluate(predictions)

print(f"AUC-ROC  : {auc:.4f}   (1.0 = perfect, 0.5 = random)")
print(f"Accuracy : {acc:.4f}")
print(f"F1 Score : {f1:.4f}")

# Confusion matrix
print("\nConfusion matrix:")
(
    predictions
    .groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
).show()

In [ ]:
# Inspect the LogisticRegression model — coefficients tell us each feature's weight
import pandas as pd

lr_model = model.stages[-1]   # last stage = fitted LogisticRegressionModel
indexer_model = model.stages[0]  # first stage = fitted StringIndexerModel

# Build feature names: [amount_f, txn_hour, category_0, category_1, ...]
feature_names = ["amount_f", "txn_hour"] + [
    f"category_{label}" for label in indexer_model.labels[:-1]  # dropLast removes one
]

coeff_df = pd.DataFrame({
    "feature":     feature_names,
    "coefficient": lr_model.coefficients.toArray(),
})
coeff_df["abs_coeff"] = coeff_df["coefficient"].abs()
print("Feature coefficients (sorted by absolute value):")
print(coeff_df.sort_values("abs_coeff", ascending=False).to_string(index=False))
print(f"\nIntercept: {lr_model.intercept:.4f}")

### Cross-Validation — Hyperparameter Tuning

`CrossValidator` finds the best hyperparameter combination by training the pipeline `numFolds` times for each combination and selecting the combination with the best average metric.

```
ParamGridBuilder builds all combinations:
  numTrees=[20, 50]  ×  maxDepth=[3, 5]  →  4 combinations
  × numFolds=3                            →  12 total fits
```

`TrainValidationSplit` is faster (single 80/20 split instead of k folds) but less robust for small datasets.

We switch to `RandomForestClassifier` here because it also produces **feature importances** — a useful interpretability tool.

In [ ]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    seed=42,
)

# Reuse the same feature engineering stages with the new classifier
rf_pipeline = Pipeline(stages=[cat_idx, cat_ohe, feat_asm, rf])

param_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [20, 50])
    .addGrid(rf.maxDepth, [3, 5])
    .build()
)
print(f"Grid combinations: {len(param_grid)}")
print(f"Total pipeline fits: {len(param_grid) * 3} (grid × numFolds)")

cv = CrossValidator(
    estimator=rf_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=auc_eval,
    numFolds=3,
    seed=42,
    parallelism=2,   # evaluate up to 2 param combos simultaneously
)

cv_model = cv.fit(train)
cv_preds  = cv_model.transform(test)
best_auc  = auc_eval.evaluate(cv_preds)

print(f"\nBest model AUC-ROC: {best_auc:.4f}")
print(f"Cross-validation AUC scores (avg per param combo): {[round(m, 4) for m in cv_model.avgMetrics]}")

# Feature importances from the best RandomForest
best_rf = cv_model.bestModel.stages[-1]
importances = pd.DataFrame({
    "feature":    feature_names[:len(best_rf.featureImportances)],
    "importance": best_rf.featureImportances.toArray(),
})
print("\nFeature Importances (best model):")
print(importances.sort_values("importance", ascending=False).to_string(index=False))

## Regression — Loan Interest Rate Prediction

Regression predicts a **continuous numeric value** rather than a class label. Here we predict a loan's `interest_rate` from the loan type, principal amount, and tenure — the inputs an underwriter uses to price a loan.

**Features:**
- `loan_type` → `StringIndexer` → `OneHotEncoder` (categorical)
- `principal` → cast to `double`
- `tenure_months` → cast to `double`

**Label**: `interest_rate` (already a `double`)

**Algorithm**: `LinearRegression` — fits a hyperplane to minimise RMSE.

**Key regression metrics:**
- **RMSE** (root mean squared error) — average prediction error in the same units as the label
- **MAE** (mean absolute error) — less sensitive to outliers than RMSE
- **R²** (coefficient of determination) — fraction of label variance explained (1.0 = perfect)

In [ ]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Prepare loan dataset
loan_df = (
    loan_accounts
    .withColumn("label",       col("interest_rate"))
    .withColumn("principal_f", col("principal").cast("double"))
    .withColumn("tenure_f",    col("tenure_months").cast("double"))
    .select("label", "principal_f", "tenure_f", "loan_type")
    .dropna()
)

print("Loan dataset summary:")
loan_df.describe("label", "principal_f", "tenure_f").show()
print("Loan types:")
loan_df.groupBy("loan_type").count().orderBy("loan_type").show()

# Build regression pipeline
loan_idx = StringIndexer(inputCol="loan_type", outputCol="loanTypeIdx", handleInvalid="keep")
loan_ohe = OneHotEncoder(inputCol="loanTypeIdx", outputCol="loanTypeVec")
loan_asm = VectorAssembler(
    inputCols=["principal_f", "tenure_f", "loanTypeVec"],
    outputCol="features",
    handleInvalid="skip",
)
lin_reg = LinearRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0,   # 0.0 = L2 (Ridge), 1.0 = L1 (Lasso)
)

reg_pipeline = Pipeline(stages=[loan_idx, loan_ohe, loan_asm, lin_reg])

loan_train, loan_test = loan_df.randomSplit([0.8, 0.2], seed=42)
reg_model  = reg_pipeline.fit(loan_train)
loan_preds = reg_model.transform(loan_test)

# Evaluate
reg_eval = RegressionEvaluator(labelCol="label", predictionCol="prediction")
rmse = reg_eval.setMetricName("rmse").evaluate(loan_preds)
mae  = reg_eval.setMetricName("mae").evaluate(loan_preds)
r2   = reg_eval.setMetricName("r2").evaluate(loan_preds)

print(f"RMSE : {rmse:.4f}   (average error in interest rate percentage points)")
print(f"MAE  : {mae:.4f}")
print(f"R²   : {r2:.4f}   (1.0 = perfect fit)")

print("\nPredicted vs Actual interest rates:")
loan_preds.select("loan_type", "principal_f", "tenure_f", "label", "prediction").show(10)

In [ ]:
# Inspect the linear regression model
lr_reg_model = reg_model.stages[-1]
loan_indexer_model = reg_model.stages[0]

reg_feature_names = ["principal_f", "tenure_f"] + [
    f"loan_type_{label}" for label in loan_indexer_model.labels[:-1]
]

reg_coeff_df = pd.DataFrame({
    "feature":     reg_feature_names,
    "coefficient": lr_reg_model.coefficients.toArray(),
})
print("LinearRegression coefficients:")
print(reg_coeff_df.to_string(index=False))
print(f"\nIntercept: {lr_reg_model.intercept:.4f}")
print(f"Training RMSE: {lr_reg_model.summary.rootMeanSquaredError:.4f}")
print(f"Training R²  : {lr_reg_model.summary.r2:.4f}")

## Saving and Loading ML Pipelines

Fitted `PipelineModel` objects are fully serialisable. All stage parameters — the `StringIndexerModel` label mapping, the `OneHotEncoderModel` category count, the `LogisticRegressionModel` coefficients — are saved together in a single directory.

| Method | Purpose |
|---|---|
| `model.write().save(path)` | Save (fails if path exists) |
| `model.write().overwrite().save(path)` | Save, overwriting existing |
| `PipelineModel.load(path)` | Load a previously saved pipeline model |
| `Pipeline.load(path)` | Load an unfitted pipeline (stages only, no learned params) |

The saved format is a directory of JSON metadata and Parquet files — human-readable and cloud-storage friendly.

In [ ]:
import tempfile
from pyspark.ml import PipelineModel

model_path = os.path.join(tempfile.gettempdir(), "fraud_lr_pipeline")

# Save the fitted pipeline model
model.write().overwrite().save(model_path)
print(f"Saved to: {model_path}")

# List the saved structure
import subprocess
result = subprocess.run(["find", model_path, "-maxdepth", "3", "-type", "f"], capture_output=True, text=True)
print("\nSaved files:")
for line in sorted(result.stdout.strip().split("\n")):
    print(" ", line)

# Load the pipeline model — no need to re-fit
loaded_model = PipelineModel.load(model_path)
print("\nLoaded stages:", [type(s).__name__ for s in loaded_model.stages])

# Verify the loaded model produces identical predictions
loaded_preds = loaded_model.transform(test)
loaded_auc   = auc_eval.evaluate(loaded_preds)
print(f"\nOriginal AUC: {auc:.6f}")
print(f"Loaded   AUC: {loaded_auc:.6f}")
print("Results match:", abs(auc - loaded_auc) < 1e-10)

In [ ]:
# Cleanup
import shutil
shutil.rmtree(model_path, ignore_errors=True)
spark.catalog.clearCache()
print("Scratch files removed.")

### Summary

**Use `spark.ml` (DataFrame-based), not `spark.mllib` (RDD-based).** The `spark.mllib` API is maintenance-only.

**Three core abstractions:**
- **Transformer** — has `transform(df) → df`. No training. Examples: `VectorAssembler`, `OneHotEncoderModel`.
- **Estimator** — has `fit(df) → Model`. Learns parameters. Examples: `StringIndexer`, `LogisticRegression`.
- **Pipeline** — chains stages. `fit()` trains all Estimator stages; `transform()` applies all stages. Returns a `PipelineModel`.

**Feature engineering:**
- `StringIndexer` — most-frequent label → index `0`. Use `handleInvalid="keep"` for unseen categories at score time.
- `OneHotEncoder` — index → sparse binary vector of length N−1 (reference level dropped).
- `VectorAssembler` — combines all feature columns into a single `features` vector. `handleInvalid="skip"` drops null rows.
- `StandardScaler` — zero mean / unit std. Use `withMean=False` for sparse (OHE) vectors.

**Classification:**
- `LogisticRegression` — standard binary classification baseline. `standardization=True` (default) normalises internally.
- `RandomForestClassifier` — ensemble of decision trees; provides `featureImportances`.
- `BinaryClassificationEvaluator` — use `areaUnderROC` for imbalanced data.
- `MulticlassClassificationEvaluator` — use `accuracy` or `f1`.

**Hyperparameter tuning:**
- `ParamGridBuilder` — define a grid of parameter combinations.
- `CrossValidator` — k-fold cross-validation over the grid. Selects the param combo with the best average metric.
- `TrainValidationSplit` — faster single-split alternative to `CrossValidator`.

**Regression:**
- `LinearRegression` — predicts continuous output. `elasticNetParam=0` → L2 (Ridge); `=1` → L1 (Lasso).
- `RegressionEvaluator` — metrics: `rmse`, `mae`, `r2`.

**Persistence:**
- `model.write().overwrite().save(path)` — saves all stage parameters.
- `PipelineModel.load(path)` — loads and is ready to `transform()` without re-fitting.